# ChakraModel — Cross-Dataset Evaluation v4 (FIXED)

**Zero-shot cross-dataset evaluation of ChakraTransformer (ViT-Large 384)**

## ⚠️ REQUIRED: Attach These Datasets Before Running
Go to the right sidebar → **Add Data** → search for each:

| # | Dataset Slug | Needed For |
|---|---|---|
| 1 | `gokulrocky/chakratransformer-weights` | Model weights (REQUIRED) |
| 2 | `gokulrocky/chakramodel-evaluation-datasets` | Kvasir-SEG test split |
| 3 | `gokulrocky/endoscene-cvc300-polyp-raw-dataset` | CVC-300 (60 images) |
| 4 | `gokulrocky/hyperkvasir-dataset-first-half-and-and-ld-dataset` | HyperKvasir + LDPolyp |
| 5 | `gokulrocky/polypdb-polyp-raw-stress-testdataset` | PolypDB 5 modalities |

> **GPU:** T4 x2 or P100. Internet: ON. Timeout: OFF.


In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "timm", "opencv-python-headless", "pandas", "tabulate"], check=True)

import os, glob, cv2, torch, json
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import timm
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️  No GPU — running on CPU (very slow)")
print(f"Device: {device}")

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}


## 🔍 Step 0: Dataset Mount Diagnostic

In [ ]:
# ── DIAGNOSTIC: Shows exactly what datasets are mounted ──────────────────────
# Run this first to verify all required datasets appear before evaluating
import os
from pathlib import Path

print("="*65)
print("  MOUNTED DATASETS IN /kaggle/input/")
print("="*65)

input_root = Path("/kaggle/input")
if not input_root.exists():
    print("ERROR: /kaggle/input not found — are you running on Kaggle?")
else:
    datasets = sorted(input_root.iterdir())
    if not datasets:
        print("⚠️  NO DATASETS ATTACHED! Please use 'Add Data' in the sidebar.")
    else:
        for ds in datasets:
            files = list(ds.rglob("*"))
            n_img = sum(1 for f in files if f.suffix.lower() in IMAGE_EXTS and f.is_file())
            n_total = sum(1 for f in files if f.is_file())
            print(f"  ✅ {ds.name:50s}  [{n_total} files, {n_img} images]")

print("="*65)

# ── Check required datasets ────────────────────────────────────────────────────
REQUIRED = {
    "chakratransformer-weights":                   "Model weights",
    "endoscene-cvc300-polyp-raw-dataset":          "EndoScene CVC-300 (60 imgs)",
}
OPTIONAL = {
    "chakramodel-evaluation-datasets":             "Kvasir-SEG test split",
    "hyperkvasir-dataset-first-half-and-and-ld-dataset": "HyperKvasir + LDPolyp",
    "polypdb-polyp-raw-stress-testdataset":        "PolypDB 5 modalities",
}

print("\nREQUIRED datasets:")
for slug, desc in REQUIRED.items():
    p = input_root / slug
    status = "✅ FOUND" if p.exists() else "❌ MISSING — attach before running!"
    print(f"  {status}: {slug} ({desc})")

print("\nOPTIONAL datasets:")
for slug, desc in OPTIONAL.items():
    p = input_root / slug
    status = "✅ Found" if p.exists() else "⚠️  Not attached (will skip)"
    print(f"  {status}: {slug} ({desc})")


## 🧠 Model: ChakraTransformerSegmenter (ViT-Large 384 + Conv Decode Head)

In [ ]:
# ── Architecture Definition ────────────────────────────────────────────────────
# This is the EXACT architecture that matches the saved checkpoint.
# Any change will cause strict=True load to fail.
#
# Architecture Summary:
#   Encoder:  ViT-Large patch16/384 (24 blocks, embed_dim=1024, 24 heads)
#             Input: 384×384×3 → patch tokens: 576×1024
#   Decoder:  Lightweight 2-stage ConvTranspose2D
#             1024 → 256 (4× upsample) → 64 (4× upsample) → 1 (conv)
#             Final bilinear resize to match input resolution
#   Total params: 309.2M (ViT-L: 307M, decoder: ~2.2M)
#   Input resolution: 384×384 (fixed — ViT position embeddings are non-flexible)

class ChakraTransformerSegmenter(nn.Module):
    def __init__(self, backbone_name='vit_large_patch16_384', pretrained=False, num_classes=1):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, features_only=False)
        self.embed_dim = self.backbone.embed_dim  # 1024 for ViT-L

        # Decoder: two ConvTranspose2d blocks (4× each = 16× total upsampling)
        # from (B, 1024, 24, 24) → (B, 1, 384, 384)
        self.decode_head = nn.Sequential(
            nn.ConvTranspose2d(self.embed_dim, 256, kernel_size=4, stride=4),  # 24→96
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(256, 64, kernel_size=4, stride=4),             # 96→384
            nn.BatchNorm2d(64),  nn.ReLU(inplace=True),
            nn.Conv2d(64, num_classes, kernel_size=3, padding=1)              # 384→384, 1ch
        )
        # Dropout2d applied after ReLU at positions 2 and 5 in decode_head
        self.dropout1 = nn.Dropout2d(p=0.5)
        self.dropout2 = nn.Dropout2d(p=0.5)

    def forward(self, x):
        B, C, H, W = x.shape  # expect 384×384
        features = self.backbone.forward_features(x)  # (B, 577, 1024) with CLS token

        # Remove CLS token if present, reshape to spatial grid
        if features.dim() == 3:
            expected_patches = (H // 16) * (W // 16)  # 576 for 384×384
            if features.shape[1] == expected_patches + 1:  # 577 → remove CLS
                features = features[:, 1:, :]
            # Reshape: (B, 576, 1024) → (B, 1024, 24, 24)
            features = features.transpose(1, 2).contiguous().view(
                B, self.embed_dim, H // 16, W // 16)

        x_dec = features
        for i, layer in enumerate(self.decode_head):
            x_dec = layer(x_dec)
            if i == 2: x_dec = self.dropout1(x_dec)   # after first ReLU
            elif i == 5: x_dec = self.dropout2(x_dec)  # after second ReLU

        # Final safety resize if needed
        if x_dec.shape[2:] != (H, W):
            x_dec = F.interpolate(x_dec, size=(H, W), mode='bilinear', align_corners=False)
        return x_dec  # (B, 1, H, W) raw logits

# ── Load Weights ───────────────────────────────────────────────────────────────
print("Initializing ChakraTransformer (ViT-Large patch16/384)...")
model = ChakraTransformerSegmenter()

# Search for checkpoint
weights_candidates = (
    glob.glob("/kaggle/input/**/chakra_transformer_best.pth", recursive=True) +
    glob.glob("/kaggle/input/**/*.pth", recursive=True) +
    glob.glob("/kaggle/input/**/*.pt", recursive=True)
)

if weights_candidates:
    wp = weights_candidates[0]
    print(f"Loading: {wp}")
    sd = torch.load(wp, map_location=device, weights_only=True)
    model.load_state_dict(sd, strict=True)
    total_params = sum(p.numel() for p in model.parameters())
    trainable   = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"✅ Weights loaded: {total_params/1e6:.1f}M params ({trainable/1e6:.1f}M trainable)")
else:
    print("❌ CRITICAL: No weights found!")
    print("   Attach 'gokulrocky/chakratransformer-weights' dataset.")
    raise FileNotFoundError("No .pth file found in /kaggle/input")

model.to(device).eval()
print(f"\n── Architecture Summary ─────────────────────────────")
print(f"   Backbone : ViT-Large patch16/384 (24 blocks, d=1024, h=16)")
print(f"   Decoder  : ConvTranspose 1024→256→64→1 (16× upsample)")
print(f"   Input    : 384×384 px (fixed resolution)")
print(f"   Output   : 384×384 binary segmentation mask")
print(f"   FP16     : AMP enabled for inference")
print(f"────────────────────────────────────────────────────")


In [ ]:
# ── Dataset + Metrics Utilities ───────────────────────────────────────────────

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

class EvalDataset(Dataset):
    MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __init__(self, img_files, mask_dict, img_size=384):
        self.files = img_files
        self.masks = mask_dict
        self.size  = img_size

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        p   = self.files[idx]
        img = cv2.imread(str(p))
        img = np.zeros((self.size, self.size, 3), dtype=np.uint8) if img is None               else cv2.resize(img, (self.size, self.size))

        mp  = self.masks.get(p.stem) or self.masks.get(p.name)
        if mp and Path(str(mp)).exists():
            msk = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)
            msk = cv2.resize(msk, (self.size, self.size), interpolation=cv2.INTER_NEAREST)                   if msk is not None else np.zeros((self.size, self.size), dtype=np.uint8)
        else:
            msk = np.zeros((self.size, self.size), dtype=np.uint8)

        t = torch.from_numpy(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)).permute(2,0,1).float() / 255.
        t = (t - self.MEAN) / self.STD
        m = torch.from_numpy((msk > 127).astype(np.float32)).unsqueeze(0)
        return t, m


def compute_metrics(model, loader):
    dices, ious, precs, recs = [], [], [], []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(device, non_blocking=True)
            with autocast(device_type='cuda', dtype=torch.float16):
                logits = model(imgs)
            probs = torch.sigmoid(logits).cpu().float().numpy()
            gts   = masks.numpy()
            for i in range(len(probs)):
                pred = (probs[i, 0] > 0.5).astype(np.float32)
                gt   = (gts[i,   0] > 0.5).astype(np.float32)
                tp = (pred * gt).sum()
                fp = (pred * (1 - gt)).sum()
                fn = ((1 - pred) * gt).sum()
                dices.append((2*tp + 1e-6) / (2*tp + fp + fn + 1e-6))
                ious.append((tp + 1e-6) / (tp + fp + fn + 1e-6))
                precs.append((tp + 1e-6) / (tp + fp + 1e-6))
                recs.append((tp + 1e-6) / (tp + fn + 1e-6))
    return {
        'dice': float(np.mean(dices)), 'std': float(np.std(dices)),
        'iou':  float(np.mean(ious)),
        'precision': float(np.mean(precs)),
        'recall':    float(np.mean(recs)),
        'n_images':  len(dices)
    }


def discover(root, img_hints=None, mask_hints=None, verbose=True):
    root = Path(root)
    if not root.exists():
        return [], {}

    IMG_HINTS  = img_hints  or ['images','image','imgs','original','frame','frames']
    MASK_HINTS = mask_hints or ['masks','mask','groundtruth','gt','ground_truth','annotations']
    IMG_HINTS  = {h.lower() for h in IMG_HINTS}
    MASK_HINTS = {h.lower() for h in MASK_HINTS}

    img_dirs, mask_dirs = [], []
    for p in root.rglob('*'):
        if not p.is_dir(): continue
        lo = p.name.lower().strip()
        if lo in IMG_HINTS:  img_dirs.append(p)
        if lo in MASK_HINTS: mask_dirs.append(p)

    if verbose and not img_dirs:
        print(f"    [discover] No img dirs found in {root}, using rglob fallback")

    img_files = []
    if img_dirs:
        for d in img_dirs:
            img_files += sorted([f for f in d.iterdir() if f.suffix.lower() in IMAGE_EXTS])
    else:
        img_files = sorted([f for f in root.rglob('*') if f.suffix.lower() in IMAGE_EXTS])

    mask_dict = {}
    if mask_dirs:
        for d in mask_dirs:
            for f in d.iterdir():
                if f.suffix.lower() in IMAGE_EXTS:
                    mask_dict[f.stem] = str(f)
                    mask_dict[f.name] = str(f)  # also index by full filename

    return img_files, mask_dict


ALL_RESULTS = {}

def run_eval(name, root, img_hints=None, mask_hints=None,
             kvasir_split=False, batch=8, note=''):
    SEP = '=' * 65
    print(f"\n{SEP}")
    print(f"  {name}")
    if note: print(f"  Note: {note}")
    print(SEP)

    img_files, mask_dict = discover(root, img_hints, mask_hints)
    if not img_files:
        print(f"  ⚠️  No images found — skipping.")
        ALL_RESULTS[name] = None
        return None

    print(f"  Images: {len(img_files)} | Masks matched: {len(mask_dict)}")

    # Optionally use the same 70/15/15 held-out test split as training
    if kvasir_split:
        np.random.seed(42)
        perm = np.random.permutation(len(img_files))
        n_tr = int(0.70 * len(img_files))
        n_ca = int(0.15 * len(img_files))
        img_files = [img_files[i] for i in perm[n_tr + n_ca:]]
        print(f"  Split:  using held-out {len(img_files)} images (seed=42, 70/15/15)")

    ds = EvalDataset(img_files, mask_dict)
    dl = DataLoader(ds, batch_size=batch, shuffle=False, num_workers=2, pin_memory=True)
    m  = compute_metrics(model, dl)

    print(f"  ✅ DSC       : {m['dice']:.4f} ± {m['std']:.4f}")
    print(f"     mIoU      : {m['iou']:.4f}")
    print(f"     Precision : {m['precision']:.4f}")
    print(f"     Recall    : {m['recall']:.4f}")
    print(f"     N images  : {m['n_images']}")
    ALL_RESULTS[name] = m
    return m

print("✅ Utilities ready.")


## 📊 Eval 1 — Kvasir-SEG Test Split (In-Distribution)

In [ ]:
# Kvasir-SEG: replicates the exact 70/15/15 split used during Combo6 training
kvasir_candidates = [
    "/kaggle/input/chakramodel-evaluation-datasets",
    "/kaggle/input/kvasir-seg",
    "/kaggle/input/kvasir-seg-data-polyp-segmentation-detection",
]
kvasir_root = next((r for r in kvasir_candidates if Path(r).exists()), None)

if kvasir_root:
    run_eval("Kvasir-SEG (held-out test)", kvasir_root,
             img_hints=['images','Images'], mask_hints=['masks','Masks'],
             kvasir_split=True,
             note="15% held-out test split, seed=42 — same as Combo6 training")
else:
    print("⚠️  Kvasir-SEG not attached. Skipping.")
    ALL_RESULTS["Kvasir-SEG (held-out test)"] = None


## 📊 Eval 2 — CVC-ClinicDB (Zero-Shot)

In [ ]:
# CVC-ClinicDB may be bundled inside the evaluation datasets or separate
# Previously scored: 0.5401 ± 0.4332 DSC
clinicdb_candidates = [
    "/kaggle/input/chakramodel-evaluation-datasets/CVC-ClinicDB",
    "/kaggle/input/chakramodel-evaluation-datasets/cvc-clinicdb",
    "/kaggle/input/cvc-clinicdb",
]
# Also search inside chakramodel-evaluation-datasets
for d in Path("/kaggle/input/chakramodel-evaluation-datasets").rglob("*") if Path("/kaggle/input/chakramodel-evaluation-datasets").exists() else []:
    if d.is_dir() and 'clinic' in d.name.lower():
        clinicdb_candidates.insert(0, str(d))

clinicdb_root = next((r for r in clinicdb_candidates if Path(r).exists()), None)

if clinicdb_root:
    run_eval("CVC-ClinicDB (zero-shot)", clinicdb_root,
             img_hints=['images','Images','Original','original'],
             mask_hints=['masks','Masks','Ground Truth','groundtruth'],
             note="612 colonoscopy frames — previously 0.5401 ± 0.4332 DSC")
else:
    print("ℹ️  CVC-ClinicDB not found as separate dataset — baseline from previous run: 0.5401 DSC")
    ALL_RESULTS["CVC-ClinicDB (zero-shot)"] = {"dice": 0.5401, "std": 0.4332, "iou": 0.4892,
                                                "precision": 0.0, "recall": 0.0, "n_images": 612,
                                                "_note": "From previous run — not re-evaluated"}
    print("  📌 Using previously recorded score: Dice=0.5401 ± 0.4332")


## 📊 Eval 3 — EndoScene CVC-300 (Zero-Shot) ← NEW

In [ ]:
# EndoScene CVC-300: 60 images
# Dataset: gokulrocky/endoscene-cvc300-polyp-raw-dataset
# Structure from Kaggle Data Explorer:
#   CVC-300/
#     images/  (60 files)
#     masks/   (60 files)

cvc300_candidates = [
    "/kaggle/input/endoscene-cvc300-polyp-raw-dataset/CVC-300",
    "/kaggle/input/endoscene-cvc300-polyp-raw-dataset",
]
cvc300_root = next((r for r in cvc300_candidates if Path(r).exists()), None)

if cvc300_root:
    # Print tree to confirm structure
    print(f"CVC-300 found at: {cvc300_root}")
    for item in sorted(Path(cvc300_root).iterdir()):
        files = list(item.iterdir()) if item.is_dir() else []
        print(f"  {item.name}/ ({len(files)} files)" if item.is_dir() else f"  {item.name}")

    run_eval("EndoScene CVC-300 (zero-shot)", cvc300_root,
             img_hints=['images','Images','image'],
             mask_hints=['masks','Masks','mask'],
             note="60 images — flat+peduncular polyp morphologies, unseen domain")
else:
    print("❌ EndoScene CVC-300 NOT FOUND!")
    print("   You MUST attach 'gokulrocky/endoscene-cvc300-polyp-raw-dataset'")
    ALL_RESULTS["EndoScene CVC-300 (zero-shot)"] = None


## 📊 Eval 4 — HyperKvasir Segmented 1K (Zero-Shot) ← NEW

In [ ]:
hk_base = "/kaggle/input/hyperkvasir-dataset-first-half-and-and-ld-dataset"

if not Path(hk_base).exists():
    print("⚠️  HyperKvasir dataset not attached — skipping.")
    ALL_RESULTS["HyperKvasir Segmented (1k)"] = None
else:
    # Print top-level structure to find the right directory
    print("HyperKvasir dataset structure:")
    for d in sorted(Path(hk_base).iterdir()):
        print(f"  {d.name}/")
        if d.is_dir():
            for sub in sorted(d.iterdir())[:6]:
                n = len(list(sub.iterdir())) if sub.is_dir() else ''
                print(f"    {sub.name}{'/' if sub.is_dir() else ''} {n}")

    # Try to find segmented polyp subset
    hk_seg_candidates = []
    for p in Path(hk_base).rglob("*"):
        if p.is_dir() and 'segmented' in p.name.lower() and ('hyper' in p.name.lower() or 'kvasir' in p.name.lower()):
            hk_seg_candidates.append(str(p))

    # Fallback: look for any dir containing both 'images' and 'masks' subdirs
    if not hk_seg_candidates:
        for p in Path(hk_base).rglob("*"):
            if p.is_dir():
                sub_names = {s.name.lower() for s in p.iterdir() if s.is_dir()}
                if 'images' in sub_names and 'masks' in sub_names:
                    hk_seg_candidates.append(str(p))

    hk_root = hk_seg_candidates[0] if hk_seg_candidates else None

    if hk_root:
        print(f"\nUsing: {hk_root}")
        run_eval("HyperKvasir Segmented (1k)", hk_root,
                 img_hints=['images','Images'],
                 mask_hints=['masks','Masks'],
                 note="1,000 segmented polyp images from HyperKvasir benchmark")
    else:
        print("⚠️  Could not locate segmented polyp subset — printing full tree:")
        for p in sorted(Path(hk_base).rglob("*"))[:40]:
            print(f"  {p.relative_to(hk_base)}")
        ALL_RESULTS["HyperKvasir Segmented (1k)"] = None


## 📊 Eval 5 — PolypDB Multi-Modality Stress Test (Zero-Shot) ← NEW

In [ ]:
pdb_root = "/kaggle/input/polypdb-polyp-raw-stress-testdataset"

if not Path(pdb_root).exists():
    print("⚠️  PolypDB not attached — skipping.")
    ALL_RESULTS["PolypDB (All Modalities)"] = None
else:
    print("PolypDB structure:")
    for d in sorted(Path(pdb_root).iterdir())[:12]:
        print(f"  {d.name}" + ("/" if d.is_dir() else ""))

    run_eval("PolypDB (All Modalities)", pdb_root,
             img_hints=['images','Images','image'],
             mask_hints=['masks','Masks','mask','annotations'],
             note="3,934 images: WLI+NBI+LCI+BLI+FICE lighting conditions")

    # Per-modality breakdown
    for modality in ['WLI', 'NBI', 'LCI', 'BLI', 'FICE']:
        mod_dir = Path(pdb_root) / modality
        if mod_dir.exists():
            run_eval(f"PolypDB ({modality})", str(mod_dir),
                     batch=16, note=f"Single optical modality: {modality}")


In [ ]:
# ETIS-Larib: previously scored 0.0000 DSC (total failure)
# No dataset attached — record the known score for paper completeness
print("ETIS-Larib: Using previously recorded score (catastrophic failure)")
ALL_RESULTS["ETIS-Larib (zero-shot)"] = {
    "dice": 0.0000, "std": 0.0000, "iou": 0.0000,
    "precision": 0.0, "recall": 0.0, "n_images": 196,
    "_note": "Total domain shift failure — recorded from previous run"
}
print("  📌 Dice=0.0000 — total generalization failure (documented in paper)")


## 📋 Final Results Summary & SOTA Comparison

In [ ]:
import json

# ── Print Results Table ────────────────────────────────────────────────────────
print("\n" + "="*80)
print("  CHAKRATRANSFORMER (ViT-LARGE 384) — FULL CROSS-DATASET EVALUATION")
print("="*80)
print(f"  {'Dataset':<40} {'Dice DSC':>12} {'mIoU':>8} {'Prec':>8} {'Rec':>8} {'N':>6}")
print("-"*80)

for name, m in ALL_RESULTS.items():
    if m is None:
        print(f"  {name:<40} {'SKIPPED':>12}")
    elif "_note" in m:
        print(f"  {name:<40} {m['dice']:.4f}±{m['std']:.4f}  {m['iou']:.4f}   (prev run)  {m['n_images']:>6}")
    else:
        print(f"  {name:<40} {m['dice']:.4f}±{m['std']:.4f}  {m['iou']:.4f}  {m['precision']:.4f}  {m['recall']:.4f}  {m['n_images']:>6}")

print("="*80)

# ── Average across evaluated datasets ─────────────────────────────────────────
valid = [(n, v) for n, v in ALL_RESULTS.items() if v is not None]
if valid:
    avg_dice = np.mean([v['dice'] for _, v in valid])
    print(f"\n  ChakraModel Avg Dice ({len(valid)} datasets): {avg_dice:.4f} ({avg_dice*100:.2f}%)")

# ── SOTA Comparison ───────────────────────────────────────────────────────────
print("\n" + "-"*80)
print("  SOTA COMPARISON (5-dataset avg: Kvasir+ClinicDB+ColonDB+CVC300+ETIS)")
print("-"*80)
sota = [
    ("PolypMamba (2025)", 89.00), ("CASCADE (2023)", 88.46),
    ("Polyp-PVT (2021)", 86.98), ("FCBFormer (2022)", 86.54),
    ("HSNet (2022)",     86.84), ("SSFormer (2022)",  85.32),
    ("PraNet (2020)",    80.10),
]
for method, score in sota:
    print(f"  {method:<25}: {score:.2f}%")
print("-"*80)

# ── Save JSON ─────────────────────────────────────────────────────────────────
out_path = "/kaggle/working/cross_dataset_results_v4.json"
saveable = {k: {kk: vv for kk, vv in v.items()} for k, v in ALL_RESULTS.items() if v is not None}
with open(out_path, 'w') as f:
    json.dump(saveable, f, indent=2)
print(f"\n✅ Results JSON saved to: {out_path}")
print("   Download this file from Kaggle output and share it for paper update.")
